In [4]:
import * as t from '@texel/color'


In [5]:
let input = {
  "name": "Tokyo Night",
  "author": "",
  "variant": "",
  "color_01": "#414868",
  "color_02": "#F7768E",
  "color_03": "#9ECE6A",
  "color_04": "#E0AF68",
  "color_05": "#7AA2F7",
  "color_06": "#BB9AF7",
  "color_07": "#7DCFFF",
  "color_08": "#A9B1D6",
  "color_09": "#414868",
  "color_10": "#F7768E",
  "color_11": "#9ECE6A",
  "color_12": "#E0AF68",
  "color_13": "#7AA2F7",
  "color_14": "#BB9AF7",
  "color_15": "#7DCFFF",
  "color_16": "#C0CAF5",
  "background": "#1A1B26",
  "foreground": "#C0CAF5",
  "cursor": "#C0CAF5",
  "hash": "92d0c154a4b8018f67d97cdf1e3cb9a76a41f7d914a12901f9225b5bbf2dd531"
}

In [17]:
import {mapEntries} from '@std/collections'
import {convert,sRGB,OKLab,hexToRGB, serialize} from '@texel/color'
import {toKebabCase} from '@std/text'
const d = Object.entries(

  mapEntries(input,([k,v]) => {
    const nk = toKebabCase(k);
    if (!v.startsWith('#')) return [nk,v];
    const rgb = hexToRGB(v);
    const lab = convert(rgb,sRGB,OKLab);
    const out = serialize(lab,OKLab);
    return [nk,out];
    
  })
).map(([k,v]) => `--color-${k}: ${v};`).join('\n')
await Deno.jupyter.display({
  "text/plain":d
},{raw:true})

--color-name: Tokyo Night;
--color-author: ;
--color-variant: ;
--color-color-01: oklab(40.94367855480772% 0.004065334730538467 -0.05440402143534995);
--color-color-02: oklab(72.26851488504042% 0.15636308511102442 0.028347643070001927);
--color-color-03: oklab(79.52627846423653% -0.08990872190675686 0.10663233698814967);
--color-color-04: oklab(78.38666090274835% 0.026597425075818415 0.10231469978808416);
--color-color-05: oklab(71.89766782347935% -0.013350596072087773 -0.131483434740149);
--color-color-06: oklab(75.14559924466643% 0.06619499005880991 -0.11700289301341882);
--color-color-07: oklab(82.00413218775158% -0.05918811085575293 -0.08681649200946251);
--color-color-08: oklab(76.65856156943876% 0.00513986666336641 -0.053453950800582795);
--color-color-09: oklab(40.94367855480772% 0.004065334730538467 -0.05440402143534995);
--color-color-10: oklab(72.26851488504042% 0.15636308511102442 0.028347643070001927);
--color-color-11: oklab(79.52627846423653% -0.08990872190675686 0.106632

In [2]:
const f = await fetch('https://genuary.art/prompts')
const b = await f.text()
b

"<!doctype html>\n" +
  '<html lang="en-US">\n' +
  "  <head>\n" +
  '    <meta charset="utf-8">\n' +
  "    <title>PROMPTS – GENUARY</title>\n" +
  '    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n' +
  '    <meta name="description" content="GENUARY is an artificially generated month of time where we build code that makes beautiful things.">\n' +
  '    <meta name="keywords" content="genuary,art,code,generative,creative,coding">\n' +
  '    <meta name="theme-color" content="#003344">\n' +
  '    <link rel="stylesheet" href="./css/style.css?v=f5b225b05e58740b1102c9b6cef52ca4b18b5605">\n' +
  '    <link rel="icon" type="image/png" href="/img/gen.png" />\n' +
  "\n" +
  "    <!-- Open Graph / Facebook -->\n" +
  '    <meta property="og:type" content="website">\n' +
  '    <meta property="og:url" content="https://genuary.art/">\n' +
  '    <meta property="og:title" content="PROMPTS – GENUARY">\n' +
  '    <meta property="og:description" content="GENUARY is an a

In [52]:
import {toKebabCase} from '@std/text'
const DayRegex = /(?<=JAN\.\s)\d+(?=\s)/i;
function processNode(node:HTMLElement) {
  const dayResult = DayRegex.exec(node.textContent);
  if (!dayResult) return null;
  const dayOfMonth = parseInt(dayResult[0],10);
  const description:string[] = [];
  const titleNode = node.nextElementSibling;
  if (!titleNode) return null;
  const title = titleNode.textContent;
  let descNode = titleNode.nextElementSibling;
  const idFull = toKebabCase(title);
  const idSections = idFull.split('-');
  const id = idSections.length > 5 ? `${dayOfMonth}-${idSections.slice(0,2).join('-')}-${idSections.at(-1)}` : `${dayOfMonth}-${idFull}`;
  while (descNode && descNode.tagName !== 'SCRIPT' && !descNode?.textContent.startsWith('JAN')) {
    description.push(descNode.textContent);
    descNode = descNode.nextElementSibling;
    
    if (descNode && descNode.tagName === 'SCRIPT') break;
  }
  return {
    day:dayOfMonth,
    prompt:title,
    title:"",
    id,
    ready:false,
    description: description.join('\n')
  }
  return [dayOfMonth,title,id, description.join('\n')]
}

In [54]:
import {JSDOM} from 'jsdom'
const dom = new JSDOM(b)
const nodes = dom.window.document.querySelectorAll('h2')
const data = Array.from(nodes).map((e) => processNode(e));
Deno.writeTextFileSync('./artworks.json',JSON.stringify({artworks:data},null,2))